In [ ]:
#
import numpy as np  # linear algebra
import os

# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd

# Visualisation
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import time
from utils.benchmarks import BENCHMARKS_TO_PATHS
from pathlib import Path


In [ ]:
%%time
### cell 0 ###

# load & cleanup
benchmark_name = "environmental-vs-ai-startups-india-eda"
file = (
    Path(BENCHMARKS_TO_PATHS[benchmark_name])
    .parent
    / "input"
    / "indian-startup-recognized-by-dpiit"
    / "Startup_Counts_Across_India.csv"
)
df = pd.read_csv(file)
factor = 3000
df = pd.concat([df] * factor)
df.info()

In [ ]:
%%time
### cell 1 ###

df.drop("S No.", axis=1, inplace=True)
df.dropna(inplace=True)
df.reset_index(inplace=True, drop=True)

# view
df.head()

In [ ]:
%%time
### cell 2 ###

env = ["Agriculture", "Green Technology", "Renewable Energy", "Waste Management"]
ai = ["AI", "Robotics", "Computer Vision"]
industry_map = {c: "ENV" for c in env}
industry_map.update({c: "AI" for c in ai})
mask = df["Industry"].isin(industry_map)
df_ea = df.loc[mask].reset_index(drop=True)
df_ea["MainIndustry"] = df_ea["Industry"].map(industry_map)
counts = df_ea["MainIndustry"].value_counts()
msg = (
    "A total of {} startups were started in India between 2016 & 2022, out of which {} are environmental related & {} are AI startups."
).format(len(df_ea), counts["ENV"], counts["AI"])
print(msg)